### 3.2.5. Nonlinear Programming

$$
\begin{aligned}
\min_{\mathbf{x}\in\mathbb{R}^n}\quad & f(\mathbf{x}) \\
\text{subject to}\quad & g_i(\mathbf{x}) \le 0,\quad i=1,\dots,m,\\
& h_j(\mathbf{x}) = 0,\quad j=1,\dots,p,
\end{aligned}
$$
with $f, g_i, h_j$ smooth and possibly nonlinear.

**Explanation:**

A nonlinear program (NLP) allows the objective or constraints to be nonlinear functions of the decision variables. When the problem is convex, any local minimizer is global; when it is nonconvex, solvers return only a local minimizer that depends on the starting point. General-purpose NLP solvers combine a smooth model of the problem with iterative methods — most prominently [sequential quadratic programming](../05_Nonlinear_Programming/11_sequential_quadratic_programming.ipynb) and interior-point methods — and the full theory is developed in the [nonlinear-programming module](../05_Nonlinear_Programming/01_general_nonlinear_programs.ipynb).

**Intuition:**

Nonlinearity bends the feasible boundary and warps the cost contours, so the optimizer must follow curved descent paths rather than straight lines.

**Numerical Example:**

Consider the Rosenbrock-type NLP with a nonlinear equality constraint:
$$
\min_{x_1,x_2}\; (1 - x_1)^2 + (x_2 - x_1^2)^2
\quad\text{subject to}\quad x_1^2 + x_2^2 = 1.
$$

The unconstrained minimizer is $(1,1)$ with value $0$, but it violates $x_1^2+x_2^2=1$. On the unit circle the objective is minimized near $(x_1,x_2)\approx(0.786, 0.618)$, giving $f\approx 0.046$. Because the constraint surface is curved, the stationarity condition
$$
\nabla f(\mathbf{x}) = \lambda\, \nabla h(\mathbf{x}),\qquad h(\mathbf{x}) = x_1^2 + x_2^2 - 1,
$$
must be solved together with feasibility rather than by setting $\nabla f = \mathbf 0$.

In [1]:
import sympy as sp

x_1, x_2, lam = sp.symbols("x_1 x_2 lambda", real=True)
objective = (1 - x_1)**2 + (x_2 - x_1**2)**2
constraint = x_1**2 + x_2**2 - 1

stationarity = [sp.diff(objective - lam*constraint, variable) for variable in (x_1, x_2)]
system = stationarity + [constraint]
solutions = sp.nsolve(system, (x_1, x_2, lam), (0.8, 0.6, 1.0))

print("stationarity residual system solved at:")
print("x_1 =", float(solutions[0]), " x_2 =", float(solutions[1]))
print("objective =", float(objective.subs({x_1: solutions[0], x_2: solutions[1]})))

stationarity residual system solved at:
x_1 = 0.8081695847299063  x_2 = 0.588949847030705
objective = 0.040919037176905815


**Equivalent casadi implementation:**

In [2]:
import casadi as ca

decision_variable = ca.SX.sym("x", 2)
objective = (1 - decision_variable[0])**2 + (decision_variable[1] - decision_variable[0]**2)**2
constraint = decision_variable[0]**2 + decision_variable[1]**2 - 1

solver = ca.nlpsol("solver", "ipopt", {"x": decision_variable, "f": objective, "g": constraint},
                   {"ipopt.print_level": 0, "print_time": 0, "ipopt.sb": "yes"})
solution = solver(x0=[0.8, 0.6], lbg=0, ubg=0)

print("minimizer =", solution["x"])
print("minimum value =", float(solution["f"]))

minimizer = [0.80817, 0.58895]
minimum value = 0.0409190371769058


**References:**

[📗 Jasour, A. (2019). *MIT 16.S498 Risk-Aware and Robust Nonlinear Planning*, Lecture 2: Nonlinear Optimization Overview.](https://ocw.mit.edu/)  
[📘 Nocedal, J., & Wright, S. J. (2006). *Numerical Optimization* (2nd ed.). Springer.](https://doi.org/10.1007/978-0-387-40065-5)

---

[⬅️ Previous: Nonsmooth Optimization](./04_nonsmooth_optimization.ipynb) | [Next: Sensitivity of Local Minima ➡️](./06_sensitivity_of_local_minima.ipynb)